# Unit17_RNN_Homework | 循環神經網路作業：時間序列預測

本作業使用模擬化工反應器溫度時間序列數據，要求學生建立 LSTM 與 GRU 模型進行預測，並比較不同模型的性能差異。

## 作業目標
- 掌握時間序列數據的正確處理流程（滑動視窗、分割、標準化）
- 理解並比較不同視窗大小對預測性能的影響
- 建立 LSTM 與 GRU 模型，並進行完整的訓練流程
- 評估模型性能，分析預測結果

## 作業說明
- 本作業共 **5 個題目**，包含程式碼填空與分析問題
- 填空題需在標記 `# TODO:` 的地方填入正確程式碼
- 分析問題需在 Markdown 區域填寫答案
- 請確認每個 cell 都能正確執行，再繳交作業

---

---
### 0. 環境設定
- 本教學課程使用Google Colab環境, 並使用TensorFlow/Keras模組建立RNN模型.
- 學生也可以使用自己的電腦, 但需配備NVIDIA GPU, 建議規格為RTX 3060或更高版本.
- 學生電腦環境安裝請參閱Part_0/Unit00_Local_Environment_Setup.ipynb.

In [ ]:
from pathlib import Path
import tensorflow as tf
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'P4_Unit17_Homework'
SOURCE_DATA_DIR = ''

try:
  from google.colab import drive
  IN_COLAB = True
  print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
  drive.mount('/content/drive', force_remount=True)
except ImportError:
  IN_COLAB = False
  print("✓ 偵測到 Local 環境")

try:
  shortcut_path = '/content/ChemE-3590'
  os.remove(shortcut_path)
except FileNotFoundError:
  pass

if IN_COLAB:
  source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3590')
  os.symlink(source_path, shortcut_path)
  shortcut_path = Path(shortcut_path)
  if source_path.exists():
    NOTEBOOK_DIR = shortcut_path / 'Part_4' / 'Unit17'
    OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    DATA_DIR = NOTEBOOK_DIR / 'data' / SOURCE_DATA_DIR
    MODEL_DIR = OUTPUT_DIR / 'models'
    FIG_DIR = OUTPUT_DIR / 'figs'
  else:
    print(f"⚠️ 找不到路徑雲端ChemE-3590路徑，請確認自己的雲端資料夾是否正確")

else:
  NOTEBOOK_DIR = Path.cwd()
  OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
  DATA_DIR = NOTEBOOK_DIR / 'data' / SOURCE_DATA_DIR
  MODEL_DIR = OUTPUT_DIR / 'models'
  FIG_DIR = OUTPUT_DIR / 'figs'

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 模型輸出目錄: {MODEL_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

# ========================================
# 檢查 GPU 狀態
# ========================================
print(f"\nTensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✓ 偵測到 GPU：{gpus[0].name}")
    print("  （RNN訓練速度將明顯快於僅用 CPU）")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("△ 未偵測到 GPU。")
    print("  訓練速度將使用 CPU（RNN在CPU上訓練較慢）")

---
### 1. 載入套件

In [ ]:
# 基礎套件
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# sklearn套件
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print(f"TensorFlow版本: {tf.__version__}")
print(f"Keras版本: {keras.__version__}")

# 設定隨機種子以確保結果可重現
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
plt.rcParams['axes.unicode_minus'] = False

---
## 一：時間序列數據準備

### 背景說明

本作業使用模擬的**化工反應器溫度時間序列**數據。該反應器的溫度訊號包含：
- 週期性波動（反應週期）
- 緩慢上升趨勢（催化劑活性衰退）
- 隨機雜訊（量測誤差）

我們的目標是：**根據過去 `window_size` 個時間步的溫度，預測下一個時間步的溫度值**（多對一回歸）。

**1.1** 請執行以下數據生成 cell（已提供），觀察時間序列的特徵，並回答下方問題。

**1.2** 請完成 `create_sequences()` 函數（填空），將一維時間序列轉換為 RNN 所需的 3D 輸入格式。

**1.3** 分別使用 `window_size = 10` 和 `window_size = 30` 生成序列數據，觀察輸出形狀的差異。

---

> **📝 分析問題 1（請在下方填寫答案）**
>
> 將 1D 時間序列轉換為 3D 輸入時，形狀為 `(n_samples, window_size, 1)`，其中 `1` 代表每個時間步的特徵數（此為單變量時間序列）。
>
> 請說明：若使用 `window_size = 10` 與 `window_size = 30`，在以下兩個方面各有何優缺點？
>
> | | window_size = 10 | window_size = 30 |
> |---|---|---|
> | 可用訓練樣本數 | ？ | ？ |
> | 捕捉長期依賴的能力 | ？ | ？ |
>
> **你的答案：** （請在此填寫）

In [ ]:
# ===== 數據生成（已提供，請直接執行）=====
def generate_reactor_temperature(n_points=2000, noise_std=0.5):
    """
    生成模擬化工反應器溫度時間序列
    
    特性：
    - 週期性波動（反應週期 = 50 時間步）
    - 緩慢上升趨勢（催化劑逐漸失活）
    - 隨機量測雜訊
    """
    t = np.arange(n_points)
    
    # 週期性成分（反應週期）
    periodic = 10 * np.sin(2 * np.pi * t / 50) + 5 * np.cos(2 * np.pi * t / 25)
    
    # 緩慢趨勢（催化劑失活，溫度逐漸升高）
    trend = 0.01 * t + 150.0  # 基準溫度 150°C，緩慢上升
    
    # 隨機雜訊
    noise = np.random.normal(0, noise_std, n_points)
    
    temperature = trend + periodic + noise
    return temperature

# 生成數據
np.random.seed(SEED)
temperature_series = generate_reactor_temperature(n_points=2000)

print(f"溫度時間序列形狀: {temperature_series.shape}")
print(f"溫度範圍: {temperature_series.min():.2f}°C ~ {temperature_series.max():.2f}°C")
print(f"平均溫度: {temperature_series.mean():.2f}°C")
print(f"標準差: {temperature_series.std():.2f}°C")

# 視覺化
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# 完整時間序列
axes[0].plot(temperature_series, linewidth=0.8, color='steelblue')
axes[0].set_title('Reactor Temperature Time Series (Full, 2000 time steps)')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Temperature (°C)')
axes[0].grid(True, alpha=0.3)

# 局部放大（前200步）
axes[1].plot(temperature_series[:200], linewidth=1.2, color='coral', marker='o', markersize=2)
axes[1].set_title('Reactor Temperature Time Series (Zoom in: first 200 steps)')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Temperature (°C)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'reactor_temperature_series.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 時間序列圖已保存")

In [ ]:
# ===== 1.2 填空題：完成 create_sequences() 函數 =====
# 說明：將一維時間序列轉換為 (n_samples, window_size, 1) 的 3D 格式

def create_sequences(data, window_size):
    """
    使用滑動視窗法將一維時間序列轉換為 RNN 輸入格式
    
    參數:
        data  : 一維時間序列 numpy array，形狀 (n_points,)
        window_size : 視窗大小（使用多少個歷史時間步作為輸入）
    
    返回:
        X : 輸入序列，形狀 (n_samples, window_size, 1)
        y : 對應的目標值（下一個時間步），形狀 (n_samples,)
    """
    X, y = [], []
    
    for i in range(len(data) - window_size):
        # TODO: 取出從 i 到 i+window_size 的數據作為一個視窗 (window_size 個點)
        window = None  # TODO: 填入正確的 slice
        
        # TODO: 目標值為視窗結束後的下一個點
        target = None  # TODO: 填入正確的索引
        
        # TODO: 將 window 轉為形狀 (window_size, 1) 並加入 X
        X.append(None)  # TODO: reshape 或直接 append
        y.append(target)
    
    return np.array(X), np.array(y)


# ===== 測試：使用兩種視窗大小 =====
WINDOW_10 = 10
WINDOW_30 = 30

X_w10, y_w10 = create_sequences(temperature_series, WINDOW_10)
X_w30, y_w30 = create_sequences(temperature_series, WINDOW_30)

print("=== 不同視窗大小的數據形狀比較 ===")
print(f"\nwindow_size = {WINDOW_10}:")
print(f"  X 形狀: {X_w10.shape}   (n_samples, window_size, 1)")
print(f"  y 形狀: {y_w10.shape}")

print(f"\nwindow_size = {WINDOW_30}:")
print(f"  X 形狀: {X_w30.shape}   (n_samples, window_size, 1)")
print(f"  y 形狀: {y_w30.shape}")

# 驗證：形狀應符合預期
assert X_w10.shape == (1990, 10, 1), f"X_w10 形狀錯誤：{X_w10.shape}，應為 (1990, 10, 1)"
assert X_w30.shape == (1970, 30, 1), f"X_w30 形狀錯誤：{X_w30.shape}，應為 (1970, 30, 1)"
print("\n✓ 形狀驗證通過！")
print(f"\n💡 觀察：使用較大視窗大小（{WINDOW_30}），可用樣本數從 {len(y_w10)} 減少為 {len(y_w30)}")

---
## 二：數據分割與標準化

本題使用 `window_size = 20` 的序列數據（已在下方生成），請完成以下工作：

**2.1** 以時間順序將數據分割為訓練集（70%）/ 驗證集（15%）/ 測試集（15%），**不可使用 `shuffle=True`**

**2.2** 對 X 和 y 分別進行標準化，注意：
  - `StandardScaler` 只能在**訓練集**上 `fit`，驗證集與測試集只做 `transform`
  - X 為 3D 張量，需先展平為 2D 再標準化，之後恢復 3D
  - y 為 1D 向量，需 reshape 後再標準化

**2.3** 執行完成後，列印各集合的大小與標準化後的統計數值

> **📝 分析問題 2（請在下方填寫答案）**
>
> 為什麼時間序列數據在分割時**不能打亂（shuffle=False）**？若打亂會造成什麼問題？
>
> **你的答案：** （請在此填寫）

In [ ]:
# 使用 window_size=20 生成序列（已提供）
WINDOW_SIZE = 20
X_all, y_all = create_sequences(temperature_series, WINDOW_SIZE)
print(f"完整序列數據形狀: X={X_all.shape}, y={y_all.shape}")

# ===== 2.1 填空題：分割數據集（不打亂！）=====
# 步驟1：先從完整數據分割出測試集（最後15%）
X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y_all,
    test_size=None,         # TODO: 填入測試集比例 (15%)
    random_state=SEED,
    shuffle=None            # TODO: 填入正確的 shuffle 參數（True 或 False）
)

# 步驟2：再從剩餘數據分割出驗證集（約佔全體15%，即剩餘85%的17.6%）
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=None,         # TODO: 填入驗證集比例，使驗證集佔全體約15%
    random_state=SEED,
    shuffle=None            # TODO: 填入正確的 shuffle 參數
)

print(f"\n=== 數據集大小 ===")
print(f"訓練集: {X_train.shape[0]} 樣本")
print(f"驗證集: {X_val.shape[0]} 樣本")
print(f"測試集: {X_test.shape[0]} 樣本")

# ===== 2.2 填空題：標準化（只在訓練集 fit！）=====
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# X 標準化：需先展平為 2D，再 reshape 回 3D
n_train, ts, nf = X_train.shape

X_train_flat = X_train.reshape(-1, nf)
X_train_scaled = None  # TODO: 在訓練集上 fit_transform，並 reshape 回 (n_train, ts, nf)

X_val_flat = X_val.reshape(-1, nf)
X_val_scaled = None    # TODO: 僅 transform（不 fit），並 reshape 回正確形狀

X_test_flat = X_test.reshape(-1, nf)
X_test_scaled = None   # TODO: 僅 transform（不 fit），並 reshape 回正確形狀

# y 標準化
y_train_scaled = None  # TODO: reshape(-1, 1) 後 fit_transform，再 flatten()
y_val_scaled   = None  # TODO: reshape(-1, 1) 後 transform，再 flatten()
y_test_scaled  = None  # TODO: reshape(-1, 1) 後 transform，再 flatten()

# ===== 2.3 驗證 =====
print(f"\n=== 標準化後統計 ===")
print(f"X_train_scaled - Mean: {X_train_scaled.mean():.4f}, Std: {X_train_scaled.std():.4f}")
print(f"y_train_scaled - Mean: {y_train_scaled.mean():.4f}, Std: {y_train_scaled.std():.4f}")
print("\n✓ 若 Mean ≈ 0、Std ≈ 1，則標準化正確！")

---
## 三：建立 LSTM 與 GRU 模型

本題要求建立兩個模型：

**3.1 建立 LSTM 模型**（架構如下）：
  - 第一層：`LSTM(64, return_sequences=True)`
  - 第二層：`LSTM(32)`
  - 第三層：`Dense(16, activation='relu')`
  - 第四層：`Dense(1)`（輸出層）
  - 輸入形狀：`input_shape=(WINDOW_SIZE, 1)`

**3.2 建立 GRU 模型**（架構如下）：
  - 第一層：`GRU(64, return_sequences=True)`
  - 第二層：`GRU(32)`
  - 第三層：`Dense(16, activation='relu')`
  - 第四層：`Dense(1)`（輸出層）
  - 輸入形狀：`input_shape=(WINDOW_SIZE, 1)`

**3.3** 列印兩個模型的 `summary()`，並比較參數量差異

> **📝 分析問題 3（請在下方填寫答案）**
>
> 在堆疊兩層 LSTM（或 GRU）時，為什麼第一層必須設定 `return_sequences=True`，而第二層不需要？
>
> 若將第一層的 `return_sequences` 設為 `False`，程式會發生什麼錯誤？請說明原因。
>
> **你的答案：** （請在此填寫）

In [ ]:
# ===== 3.1 填空題：建立 LSTM 模型 =====
# input_shape = (WINDOW_SIZE, 1)

model_lstm = Sequential([
    # TODO: 第一層 LSTM，64 個單元，返回完整序列，指定輸入形狀
    
    # TODO: 第二層 LSTM，32 個單元，不返回序列
    
    # TODO: Dense 隱藏層，16 個神經元，relu 激活
    
    # TODO: 輸出層，1 個神經元，無激活函數（回歸）
    
], name='LSTM_Model')

print("=== LSTM 模型架構 ===")
model_lstm.summary()
print(f"\n✓ LSTM 總參數量: {model_lstm.count_params():,}")

In [ ]:
# ===== 3.2 填空題：建立 GRU 模型 =====
# 架構與 LSTM 相同，但將 LSTM 層替換為 GRU 層

model_gru = Sequential([
    # TODO: 第一層 GRU，64 個單元，返回完整序列，指定輸入形狀
    
    # TODO: 第二層 GRU，32 個單元，不返回序列
    
    # TODO: Dense 隱藏層，16 個神經元，relu 激活
    
    # TODO: 輸出層，1 個神經元，無激活函數（回歸）
    
], name='GRU_Model')

print("=== GRU 模型架構 ===")
model_gru.summary()
print(f"\n✓ GRU 總參數量: {model_gru.count_params():,}")

# ===== 3.3 比較兩模型參數量 =====
lstm_params = model_lstm.count_params()
gru_params  = model_gru.count_params()
ratio = lstm_params / gru_params

print(f"\n=== 參數量比較 ===")
print(f"LSTM 參數量: {lstm_params:,}")
print(f"GRU  參數量: {gru_params:,}")
print(f"LSTM/GRU 參數比: {ratio:.2f}x")
print(f"\n💡 理論預期：LSTM 參數量約為 GRU 的 4/3 = 1.33 倍（因 LSTM 有4個門，GRU 有3個）")

---
## 四：模型訓練與 Callbacks 設定

本題要求對 LSTM 與 GRU 模型進行編譯與訓練。

**4.1** 請完成模型的編譯設定：
  - 優化器：`Adam`，學習率 `0.001`，並設定 **梯度裁剪** `clipnorm=1.0`
  - 損失函數：`'mse'`
  - 評估指標：`['mae']`

**4.2** 請設定以下三個 Callbacks：
  - `EarlyStopping`：監控 `val_loss`，patience=20，並在停止時恢復最佳權重
  - `ModelCheckpoint`：只保存最佳模型（`save_best_only=True`）
  - `ReduceLROnPlateau`：patience=10，factor=0.5，min_lr=1e-7

**4.3** 使用以下參數訓練兩個模型：
  - `epochs=150`
  - `batch_size=32`
  - `shuffle=False`（時間序列不打亂！）
  - 提供驗證集 `(X_val_scaled, y_val_scaled)`

> **📝 分析問題 4（請在下方填寫答案）**
>
> 1. 在訓練 RNN 模型時，為何需要在 `Adam` 優化器中設定 `clipnorm=1.0`？這個設定解決了什麼問題？
>
> 2. `EarlyStopping` 的 `restore_best_weights=True` 有什麼作用？若設為 `False` 會有什麼風險？
>
> **你的答案：** （請在此填寫）

In [ ]:
# ===== 4.1 & 4.2 填空題：編譯 LSTM 模型並設定 Callbacks =====
print("編譯 LSTM 模型...")

# 編譯 LSTM
model_lstm.compile(
    optimizer=None,   # TODO: Adam(learning_rate=0.001, clipnorm=1.0)
    loss=None,        # TODO: 回歸任務損失函數
    metrics=None      # TODO: 評估指標
)

# Callbacks for LSTM
callbacks_lstm = [
    EarlyStopping(
        monitor=None,               # TODO: 監控驗證損失
        patience=None,              # TODO: 填入 patience 值
        restore_best_weights=None,  # TODO: 恢復最佳權重？
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_DIR / 'best_lstm_model.keras'),
        monitor='val_loss',
        save_best_only=None,        # TODO: 只保存最佳？
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=None,                # TODO: 學習率縮減倍數
        patience=None,              # TODO: patience 值
        min_lr=None,                # TODO: 最小學習率
        verbose=1
    )
]

print("✓ LSTM 模型編譯完成")

In [ ]:
# ===== 4.1 & 4.2 填空題：編譯 GRU 模型並設定 Callbacks =====
print("編譯 GRU 模型...")

# 編譯 GRU（與 LSTM 設定相同）
model_gru.compile(
    optimizer=None,   # TODO: 與 LSTM 相同的 Adam 設定
    loss=None,        # TODO: 損失函數
    metrics=None      # TODO: 評估指標
)

# Callbacks for GRU
callbacks_gru = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_DIR / 'best_gru_model.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    )
]

print("✓ GRU 模型編譯完成")

In [ ]:
# ===== 4.3 填空題：訓練 LSTM 模型 =====
print("開始訓練 LSTM 模型...\n")

history_lstm = model_lstm.fit(
    X_train_scaled, y_train_scaled,
    epochs=None,                              # TODO: 填入 epochs 數
    batch_size=None,                          # TODO: 填入 batch_size
    validation_data=(X_val_scaled, y_val_scaled),
    callbacks=callbacks_lstm,
    shuffle=None,                             # TODO: 時間序列應為 True 還是 False？
    verbose=1
)

print("\n✓ LSTM 訓練完成！")

In [ ]:
# ===== 4.3 填空題：訓練 GRU 模型 =====
print("開始訓練 GRU 模型...\n")

history_gru = model_gru.fit(
    X_train_scaled, y_train_scaled,
    epochs=150,
    batch_size=32,
    validation_data=(X_val_scaled, y_val_scaled),
    callbacks=callbacks_gru,
    shuffle=False,
    verbose=1
)

print("\n✓ GRU 訓練完成！")

In [ ]:
# ===== 訓練曲線視覺化（已提供，請直接執行）=====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History: LSTM vs GRU', fontsize=15, fontweight='bold')

histories = {'LSTM': history_lstm, 'GRU': history_gru}
colors    = {'LSTM': 'coral', 'GRU': 'lightgreen'}

for idx, (name, hist) in enumerate(histories.items()):
    ax = axes[idx]
    ax.plot(hist.history['loss'],     label='Train Loss', color=colors[name], linewidth=2, alpha=0.8)
    ax.plot(hist.history['val_loss'], label='Val Loss',   color=colors[name], linewidth=2, linestyle='--')
    
    best_epoch = np.argmin(hist.history['val_loss'])
    best_loss  = hist.history['val_loss'][best_epoch]
    ax.scatter([best_epoch], [best_loss], color='red', s=100, zorder=5)
    ax.annotate(f'Best\nEpoch {best_epoch+1}\nLoss={best_loss:.4f}',
                xy=(best_epoch, best_loss), xytext=(10, 10),
                textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.4', fc='yellow', alpha=0.7), fontsize=9)
    
    ax.set_title(f'{name} Training History')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'training_history_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 收斂速度分析:")
for name, hist in histories.items():
    best_ep   = np.argmin(hist.history['val_loss']) + 1
    total_ep  = len(hist.history['loss'])
    print(f"  {name:6s}: 最佳 Epoch={best_ep:3d}, 總訓練 Epoch={total_ep:3d}")

---
## 五：性能評估、比較與分析

本題要求完成模型評估並進行視覺化分析。

**5.1** 在**測試集**上評估兩個模型，計算以下指標（需對預測結果進行**反標準化**）：
  - MAE（平均絕對誤差）
  - RMSE（均方根誤差）
  - R²（決定係數）

**5.2** 繪製下列圖表：
  - 前 100 個測試樣本的「預測值 vs. 實際值」時序對比圖（LSTM 與 GRU 各一條）
  - LSTM 與 GRU 的 MAE / RMSE / R² 指標柱狀比較圖
  - 散點圖（Predicted vs Actual），觀察偏差模式

**5.3** 儲存最佳模型與 Scaler

> **📝 分析問題 5（請在下方填寫答案）**
>
> 根據實驗結果，請回答以下問題：
>
> 1. LSTM 與 GRU 在本任務上的性能差異如何？哪個模型表現更好？為什麼？
>
> 2. 觀察散點圖，預測值與實際值的分布是否均勻分布在對角線兩側？若有系統性偏差（bias），可能是什麼原因造成的？
>
> 3. 若要進一步提升模型性能，你會嘗試哪些調整方法？（至少列出 2 個）
>
> **你的答案：** （請在此填寫）

In [ ]:
# ===== 5.1 填空題：評估兩個模型（測試集，原始尺度）=====

def evaluate_model(model, X_test_sc, y_test_sc, scaler_y, model_name):
    """
    評估模型性能並回傳指標字典
    - 預測結果需反標準化後再計算指標
    """
    # TODO: 使用 model.predict() 取得標準化預測值
    y_pred_scaled = None  # TODO
    
    # TODO: 使用 scaler_y.inverse_transform() 反標準化
    y_pred = None  # TODO: inverse_transform 後 flatten()
    y_true = None  # TODO: y_test_sc 也需反標準化 flatten()
    
    # TODO: 計算評估指標
    mae  = None  # TODO: mean_absolute_error(y_true, y_pred)
    rmse = None  # TODO: np.sqrt(mean_squared_error(...))
    r2   = None  # TODO: r2_score(y_true, y_pred)
    
    print(f"\n=== {model_name} 測試集性能（原始尺度）===")
    print(f"  MAE  : {mae:.4f} °C")
    print(f"  RMSE : {rmse:.4f} °C")
    print(f"  R²   : {r2:.4f}")
    
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'y_pred': y_pred, 'y_true': y_true}


# 評估兩個模型
results_lstm = evaluate_model(model_lstm, X_test_scaled, y_test_scaled, scaler_y, 'LSTM')
results_gru  = evaluate_model(model_gru,  X_test_scaled, y_test_scaled, scaler_y, 'GRU')

In [ ]:
# ===== 5.2 填空題：視覺化結果 =====

fig = plt.figure(figsize=(18, 12))
gs  = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

results_all = {'LSTM': results_lstm, 'GRU': results_gru}
colors      = {'LSTM': 'coral', 'GRU': 'lightgreen'}

# ----- 子圖1（橫跨兩格）：前100個測試樣本的時序對比 -----
ax_ts = fig.add_subplot(gs[0, :2])
n_show = 100
ax_ts.plot(results_lstm['y_true'][:n_show], label='Actual', color='black', linewidth=2, zorder=3)
for name, res in results_all.items():
    ax_ts.plot(res['y_pred'][:n_show], label=f'{name} Predicted',
               color=colors[name], linewidth=1.5, linestyle='--', alpha=0.85)
ax_ts.set_title(f'Predictions vs Actual (First {n_show} Test Samples)')
ax_ts.set_xlabel('Sample Index')
ax_ts.set_ylabel('Temperature (°C)')
ax_ts.legend()
ax_ts.grid(True, alpha=0.3)

# ----- 子圖2：指標柱狀比較 -----
ax_bar = fig.add_subplot(gs[0, 2])
metrics_names = ['MAE (°C)', 'RMSE (°C)']
x = np.arange(len(metrics_names))
width = 0.35
for i, (name, res) in enumerate(results_all.items()):
    vals = [res['mae'], res['rmse']]
    bars = ax_bar.bar(x + i * width, vals, width, label=name, color=colors[name], alpha=0.8)
    for bar, v in zip(bars, vals):
        ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{v:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax_bar.set_xticks(x + width / 2)
ax_bar.set_xticklabels(metrics_names)
ax_bar.set_title('MAE & RMSE Comparison')
ax_bar.legend()
ax_bar.grid(axis='y', alpha=0.3)

# ----- 子圖3 & 4：散點圖（LSTM / GRU）-----
for idx, (name, res) in enumerate(results_all.items()):
    ax_sc = fig.add_subplot(gs[1, idx])
    ax_sc.scatter(res['y_true'], res['y_pred'], alpha=0.5, s=15, color=colors[name])
    lim_min = min(res['y_true'].min(), res['y_pred'].min()) - 1
    lim_max = max(res['y_true'].max(), res['y_pred'].max()) + 1
    ax_sc.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=2, label='Perfect Prediction')
    ax_sc.set_title(f'{name} – Predicted vs Actual\nR²={res["r2"]:.4f}')
    ax_sc.set_xlabel('Actual Temperature (°C)')
    ax_sc.set_ylabel('Predicted Temperature (°C)')
    ax_sc.legend(fontsize=8)
    ax_sc.grid(True, alpha=0.3)

# ----- 子圖5：R² 比較 -----
ax_r2 = fig.add_subplot(gs[1, 2])
r2_vals  = [res['r2'] for res in results_all.values()]
bar_cols = [colors[n] for n in results_all.keys()]
bars = ax_r2.bar(list(results_all.keys()), r2_vals, color=bar_cols, alpha=0.8)
for bar, v in zip(bars, r2_vals):
    ax_r2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
               f'{v:.4f}', ha='center', va='bottom', fontweight='bold')
ax_r2.set_title('R² Score Comparison')
ax_r2.set_ylabel('R²')
ax_r2.set_ylim([min(r2_vals) * 0.98, 1.0])
ax_r2.grid(axis='y', alpha=0.3)

fig.suptitle('LSTM vs GRU: Comprehensive Performance Comparison', fontsize=16, fontweight='bold')
plt.savefig(FIG_DIR / 'performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 性能比較圖已保存")

In [ ]:
# ===== 5.3 保存最佳模型與 Scaler（已提供，請直接執行）=====

# 根據 R² 選出最佳模型
best_name  = max(results_all.keys(), key=lambda n: results_all[n]['r2'])
best_model = model_lstm if best_name == 'LSTM' else model_gru

# 保存最佳模型
best_model_path = MODEL_DIR / f'homework_best_{best_name.lower()}_model.keras'
best_model.save(best_model_path)
print(f"✓ 最佳模型 ({best_name}) 已保存至: {best_model_path}")

# 保存 Scalers
scaler_path = MODEL_DIR / 'homework_scalers.pkl'
joblib.dump({'scaler_X': scaler_X, 'scaler_y': scaler_y}, scaler_path)
print(f"✓ Scalers 已保存至: {scaler_path}")

# 保存比較結果
comparison_data = {name: {k: v for k, v in res.items() if k != 'y_pred' and k != 'y_true'}
                   for name, res in results_all.items()}
joblib.dump(comparison_data, MODEL_DIR / 'homework_comparison.pkl')
print(f"✓ 比較結果已保存")

# 列印最終比較表
print(f"\n{'='*55}")
print(f"{'Model':<8} {'MAE (°C)':>10} {'RMSE (°C)':>11} {'R²':>8}")
print(f"{'-'*55}")
for name, res in results_all.items():
    marker = " ◀ Best" if name == best_name else ""
    print(f"{name:<8} {res['mae']:>10.4f} {res['rmse']:>11.4f} {res['r2']:>8.4f}{marker}")
print(f"{'='*55}")

---
## 作業總結

恭喜完成本次作業！請確認以下所有項目均已完成：

### ✅ 繳交前確認清單

**程式碼部分**:
- [ ] 題目一：`create_sequences()` 函數填空完成，輸出形狀驗證通過
- [ ] 題目二：數據分割（`shuffle=False`）與標準化（僅訓練集 `fit`）正確完成
- [ ] 題目三：LSTM 模型架構建立完整（含 `return_sequences` 正確設定）
- [ ] 題目三：GRU 模型架構建立完整
- [ ] 題目四：LSTM 模型編譯（Adam + `clipnorm`）與 Callbacks 設定完整
- [ ] 題目四：兩模型訓練完成（`shuffle=False`，`epochs=150`）
- [ ] 題目五：`evaluate_model()` 函數填空完成（反標準化 + 指標計算）
- [ ] 所有 cells 均可從頭到尾依序執行，無錯誤

**分析問題部分**:
- [ ] 分析問題 1：視窗大小比較表已填寫
- [ ] 分析問題 2：時間序列不可 shuffle 的原因已說明
- [ ] 分析問題 3：`return_sequences` 的作用已解釋
- [ ] 分析問題 4：梯度裁剪與 `restore_best_weights` 的作用已說明
- [ ] 分析問題 5：LSTM vs GRU 性能差異分析、偏差原因、改善方法已填寫

---

**課程資訊**
- 課程名稱：AI在化工上之應用 (ChemE 3590)
- 課程單元：Unit17 - RNN Overview 循環神經網路概論
- 課程製作：逢甲大學 化工系 智慧程序系統工程實驗室
- 授課教師：莊曜禎 助理教授
- 更新日期：2026-06-18

**課程授權 [CC BY-NC-SA 4.0]**
 - 本教材遵循 [創用CC 姓名標示-非商業性-相同方式分享 4.0 國際 (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.zh) 授權。

---